# HW3 sentiment classification

Train three uncased BERT specialists and combine their probabilities with equal-weight soft voting. Report held-out and pooled scores on the frozen split.

In [2]:
%%bash
set -e
cd /content

if [ -d Intro_to_AI_HW/.git ]; then
    git -C Intro_to_AI_HW pull --ff-only origin main
else
    git clone --branch main https://github.com/billlllllllly/Intro_to_AI_HW.git
fi

cd Intro_to_AI_HW/IAI_HW3
pip install -r requirements.txt

Already up to date.


From https://github.com/billlllllllly/Intro_to_AI_HW
 * branch            main       -> FETCH_HEAD


The previous fold-bagged run reported cross-calibrated OOF macro-F1 0.8830 and calibrated held-out test macro-F1 0.8690 (8,105 development and 895 test tweets). The next cell computes pooled macro-F1 on the same validation and test rows used for the specialist comparison when the original checkpoints are available.

In [3]:
%cd /content/Intro_to_AI_HW/IAI_HW3
from baseline_evaluation import baseline_score
baseline = baseline_score("/content/hw3_fold_checkpoints")


/content/Intro_to_AI_HW/IAI_HW3
Exact pooled baseline unavailable: previous fold checkpoints are missing.


In [4]:
import re
import pandas as pd

tweets = pd.read_csv("dataset/dataset.csv", keep_default_na=False)["text"]
patterns = {"emoticon": r"(?<!\w)(?:[:=;]-?[()\[\]DP/]|<3|\^_?\^|-_-)(?!\w)",
            "hashtag": r"#\w+", "elongated": r"([A-Za-z])\1{2,}", "HTML entity": r"&\w+;"}
for name, pattern in patterns.items():
    count = tweets.map(re.compile(pattern).search).notna().sum()
    print(f"{name}: {count}/{len(tweets)} ({count/len(tweets):.2%})")


emoticon: 178/9000 (1.98%)
hashtag: 1564/9000 (17.38%)
elongated: 56/9000 (0.62%)
HTML entity: 475/9000 (5.28%)


In [ ]:
%cd /content/Intro_to_AI_HW/IAI_HW3
import torch
from main import run

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training.")

predictions, histories = run(
    ensemble=True, epochs=3, checkpoint_dir="/content/hw3_clean_specialists",
    review_path="/content/neutral_errors.csv",
)
if baseline is not None:
    print(f"Pooled macro-F1 change: {predictions.attrs['pooled_macro_f1'] - baseline:+.6f}")


In [ ]:
import pandas as pd

errors = pd.read_csv('/content/neutral_errors.csv')
display(errors.groupby(['label', 'ensemble']).size().rename('count').to_frame())
display(errors.groupby('label', group_keys=False).head(10))

In [ ]:
from google.colab import files

output_path = "/content/predictions.csv"
predictions.to_csv(output_path, index=False)
files.download(output_path)

In [ ]:
import matplotlib.pyplot as plt

for name, history in histories.items():
    epochs = range(1, len(history) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(9, 4))
    figure.suptitle(name)
    for index, title in enumerate(("Cross-entropy", "Macro-F1")):
        axes[index].plot(epochs, [train[index] for train, _ in history], label="Train")
        axes[index].plot(epochs, [val[index] for _, val in history], label="Validation")
        axes[index].set(xlabel="Epoch", title=title, xticks=list(epochs))
        axes[index].legend()
    figure.tight_layout()
    plt.show()